# Estruturação da Camada Gold

Nesta etapa iremos definir nossas tabelas fato e dimensão no modelo star schema, e criar algumas visualizacões agregadas para facilitar as analises posteriores.

In [0]:
vra = spark.table("mvp_engenharia_dados.silver.silver_vra")
empresas = spark.table("mvp_engenharia_dados.silver.silver_empresas")
aerodromos = spark.table("mvp_engenharia_dados.silver.silver_aerodromos")

print(f"silver_vra:       {vra.count():,d} registros")
print(f"silver_empresas:   {empresas.count()} registros")
print(f"silver_aerodromos: {aerodromos.count()} registros")

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

dim_empresas = (empresas
    .withColumn("sk_empresa", monotonically_increasing_id())
    .select(
        "sk_empresa",
        col("icao").alias("sigla_icao_empresa"),
        "empresa",
        "razao",
        "cidade",
        "uf",
        "ativa",
        "estrangeira",
    ))

(dim_empresas.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.gold.dim_empresas"))

print(f"dim_empresas: {dim_empresas.count()} registros")
display(dim_empresas.limit(10))

In [0]:
dim_aerodromos = (aerodromos
    .withColumn("sk_aerodromo", monotonically_increasing_id())
    .select(
        "sk_aerodromo",
        col("codigo_oaci").alias("sigla_icao_aeroporto"),
        "aerodromo",
        "nome",
        "municipio",
        "uf",
        "latgeopoint",
        "longeopoint",
        "altitude",
        "situacao",
    ))

(dim_aerodromos.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.gold.dim_aerodromos"))

print(f"dim_aerodromos: {dim_aerodromos.count()} registros")
display(dim_aerodromos.limit(10))

In [0]:
from pyspark.sql.functions import (
    explode, sequence, to_date, lit, col, year, month, dayofmonth,
    dayofweek, date_format, quarter, weekofyear, when, concat
)

min_data = vra.selectExpr("LEAST(MIN(data_partida), MIN(data_chegada))").collect()[0][0]
max_data = vra.selectExpr("GREATEST(MAX(data_partida), MAX(data_chegada))").collect()[0][0]

# Garante cobertura até o último dia do mês da data máxima
max_data_ajustada = spark.sql(f"""
    SELECT last_day('{max_data}') AS d
""").collect()[0][0]

calendario = (
    spark.range(1)
    .select(explode(
        sequence(to_date(lit(str(min_data))), to_date(lit(str(max_data_ajustada))))
    ).alias("data"))
)

dim_calendario = (
    calendario
    .withColumn("sk_data", date_format(col("data"), "yyyyMMdd").cast("int"))
    .withColumn("ano", year(col("data")))
    .withColumn("mes", month(col("data")))
    .withColumn("nome_mes",
        when(month(col("data")) == 1, "Janeiro")
        .when(month(col("data")) == 2, "Fevereiro")
        .when(month(col("data")) == 3, "Março")
        .when(month(col("data")) == 4, "Abril")
        .when(month(col("data")) == 5, "Maio")
        .when(month(col("data")) == 6, "Junho")
        .when(month(col("data")) == 7, "Julho")
        .when(month(col("data")) == 8, "Agosto")
        .when(month(col("data")) == 9, "Setembro")
        .when(month(col("data")) == 10, "Outubro")
        .when(month(col("data")) == 11, "Novembro")
        .when(month(col("data")) == 12, "Dezembro"))
    .withColumn("ano_mes", date_format(col("data"), "yyyy-MM"))
    .withColumn("dia", dayofmonth(col("data")))
    .withColumn("dia_semana", dayofweek(col("data")))
    .withColumn("nome_dia_semana",
        when(dayofweek(col("data")) == 1, "Domingo")
        .when(dayofweek(col("data")) == 2, "Segunda-feira")
        .when(dayofweek(col("data")) == 3, "Terça-feira")
        .when(dayofweek(col("data")) == 4, "Quarta-feira")
        .when(dayofweek(col("data")) == 5, "Quinta-feira")
        .when(dayofweek(col("data")) == 6, "Sexta-feira")
        .when(dayofweek(col("data")) == 7, "Sábado"))
    .withColumn("trimestre", quarter(col("data")))
    .withColumn("nome_trimestre", concat(lit("T"), quarter(col("data")), lit(" "), year(col("data"))))
    .withColumn("semana_ano", weekofyear(col("data")))
    .withColumn("fim_de_semana", dayofweek(col("data")).isin(1, 7))
    .withColumn("dia_util", ~dayofweek(col("data")).isin(1, 7))
)

(dim_calendario.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.gold.dim_calendario"))

print(f"dim_calendario: {dim_calendario.count()} registros")
display(dim_calendario.limit(10))

In [0]:
from pyspark.sql.functions import monotonically_increasing_id, col

fato_voos = (vra
    .join(dim_empresas, vra.sigla_icao_empresa_aerea == dim_empresas.sigla_icao_empresa, "inner")
    .join(dim_aerodromos.alias("aero_orig"), vra.sigla_icao_aeroporto_origem == col("aero_orig.sigla_icao_aeroporto"), "inner")
    .join(dim_aerodromos.alias("aero_dest"), vra.sigla_icao_aeroporto_destino == col("aero_dest.sigla_icao_aeroporto"), "inner")
    .join(dim_calendario.alias("cal"), vra.data_partida == col("cal.data"), "inner")
    .join(dim_calendario.alias("cal_cheg"), vra.data_chegada == col("cal_cheg.data"), "inner")
    .withColumn("sk_voo", monotonically_increasing_id())
    .withColumn("pontual_partida", col("situacao_partida").isin("Antecipado", "Pontual"))
    .withColumn("pontual_chegada", col("situacao_chegada").isin("Antecipado", "Pontual"))
    .select(
        "sk_voo",
        "sk_empresa",
        col("cal.sk_data").alias("sk_data_partida"),
        col("cal_cheg.sk_data").alias("sk_data_chegada"),
        col("aero_orig.sk_aerodromo").alias("sk_aeroporto_origem"),
        col("aero_dest.sk_aerodromo").alias("sk_aeroporto_destino"),
        "numero_voo",
        "modelo_equipamento",
        "numero_de_assentos",
        "partida_prevista",
        "partida_real",
        "chegada_prevista",
        "chegada_real",
        "situacao_partida",
        "situacao_chegada",
        "pontual_partida",
        "pontual_chegada",
        "atraso_partida_min",
        "atraso_chegada_min",
        "descricao_etapa",
        "referencia",
    ))

(fato_voos.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("mvp_engenharia_dados.gold.fato_voos"))

print(f"fato_voos: {fato_voos.count():,d} registros")
display(fato_voos.limit(10))

### Constraints (PK e FK)

O Unity Catalog exige `NOT NULL` em colunas de `PRIMARY KEY` e `FOREIGN KEY`. Como `monotonically_increasing_id()` gera colunas nullable no Delta, aplicamos `ALTER COLUMN ... SET NOT NULL` antes de criar as constraints.

**Primary Keys:** `sk_empresa` (dim_empresas), `sk_aerodromo` (dim_aerodromos), `sk_data` (dim_calendario), `sk_voo` (fato_voos)

**Foreign Keys em fato_voos:**
- `sk_empresa` → `dim_empresas.sk_empresa`
- `sk_aeroporto_origem` → `dim_aerodromos.sk_aerodromo`
- `sk_aeroporto_destino` → `dim_aerodromos.sk_aerodromo`
- `sk_data_partida` → `dim_calendario.sk_data`
- `sk_data_chegada` → `dim_calendario.sk_data`

In [0]:
%sql
ALTER TABLE mvp_engenharia_dados.gold.dim_empresas ALTER COLUMN sk_empresa SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.dim_aerodromos ALTER COLUMN sk_aerodromo SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.dim_calendario ALTER COLUMN sk_data SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos ALTER COLUMN sk_voo SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos ALTER COLUMN sk_empresa SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos ALTER COLUMN sk_data_partida SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos ALTER COLUMN sk_data_chegada SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos ALTER COLUMN sk_aeroporto_origem SET NOT NULL;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos ALTER COLUMN sk_aeroporto_destino SET NOT NULL;

ALTER TABLE mvp_engenharia_dados.gold.dim_empresas DROP CONSTRAINT IF EXISTS pk_dim_empresas;
ALTER TABLE mvp_engenharia_dados.gold.dim_aerodromos DROP CONSTRAINT IF EXISTS pk_dim_aerodromos;
ALTER TABLE mvp_engenharia_dados.gold.dim_calendario DROP CONSTRAINT IF EXISTS pk_dim_calendario;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos DROP CONSTRAINT IF EXISTS pk_fato_voos;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos DROP CONSTRAINT IF EXISTS fk_fato_empresa;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos DROP CONSTRAINT IF EXISTS fk_fato_aeroporto_origem;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos DROP CONSTRAINT IF EXISTS fk_fato_aeroporto_destino;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos DROP CONSTRAINT IF EXISTS fk_fato_data_partida;
ALTER TABLE mvp_engenharia_dados.gold.fato_voos DROP CONSTRAINT IF EXISTS fk_fato_data_chegada;

ALTER TABLE mvp_engenharia_dados.gold.dim_empresas
  ADD CONSTRAINT pk_dim_empresas PRIMARY KEY (sk_empresa);

ALTER TABLE mvp_engenharia_dados.gold.dim_aerodromos
  ADD CONSTRAINT pk_dim_aerodromos PRIMARY KEY (sk_aerodromo);

ALTER TABLE mvp_engenharia_dados.gold.dim_calendario
  ADD CONSTRAINT pk_dim_calendario PRIMARY KEY (sk_data);

ALTER TABLE mvp_engenharia_dados.gold.fato_voos
  ADD CONSTRAINT pk_fato_voos PRIMARY KEY (sk_voo);

ALTER TABLE mvp_engenharia_dados.gold.fato_voos
  ADD CONSTRAINT fk_fato_empresa FOREIGN KEY (sk_empresa)
  REFERENCES mvp_engenharia_dados.gold.dim_empresas(sk_empresa);

ALTER TABLE mvp_engenharia_dados.gold.fato_voos
  ADD CONSTRAINT fk_fato_aeroporto_origem FOREIGN KEY (sk_aeroporto_origem)
  REFERENCES mvp_engenharia_dados.gold.dim_aerodromos(sk_aerodromo);

ALTER TABLE mvp_engenharia_dados.gold.fato_voos
  ADD CONSTRAINT fk_fato_aeroporto_destino FOREIGN KEY (sk_aeroporto_destino)
  REFERENCES mvp_engenharia_dados.gold.dim_aerodromos(sk_aerodromo);

ALTER TABLE mvp_engenharia_dados.gold.fato_voos
  ADD CONSTRAINT fk_fato_data_partida FOREIGN KEY (sk_data_partida)
  REFERENCES mvp_engenharia_dados.gold.dim_calendario(sk_data);

ALTER TABLE mvp_engenharia_dados.gold.fato_voos
  ADD CONSTRAINT fk_fato_data_chegada FOREIGN KEY (sk_data_chegada)
  REFERENCES mvp_engenharia_dados.gold.dim_calendario(sk_data);

In [0]:
%sql

COMMENT ON TABLE mvp_engenharia_dados.gold.dim_empresas IS 'Dimensão de empresas aéreas que operam voos domésticos no Brasil. Originada da silver_empresas, complementada com PTB (Voepass) e PAM (MAP) e com remoção de empresas exclusivamente de carga (MWM, BPC, TOT, TTL, LTG) no notebook 04-camada-silver.';

COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.sk_empresa IS 'Surrogate key da empresa aérea (gerada por monotonically_increasing_id)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.sigla_icao_empresa IS 'Designador ICAO de 3 letras da empresa aérea (chave natural)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.empresa IS 'Primeira palavra da razão social no formato PRIMEIRA_PALAVRA (ICAO), ex: GOL (GLO), TOTAL (TTL)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.razao IS 'Razão social da empresa aérea';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.cidade IS 'Cidade onde a empresa é sediada';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.uf IS 'Unidade da federação da sede da empresa';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.ativa IS 'Status da empresa no cadastro da ANAC (ATIVA/INATIVA)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_empresas.estrangeira IS 'Código designador internacional da empresa (ex: WD, 2S, TT). Valor "-" quando a empresa não possui designador internacional';

COMMENT ON TABLE mvp_engenharia_dados.gold.dim_aerodromos IS 'Dimensão de aeródromos brasileiros. Originada da silver_aerodromos, complementada com aeródromos do SIROS no notebook 04-camada-silver. Aeródromos estrangeiros foram removidos pois a análise é doméstica.';

COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.sk_aerodromo IS 'Surrogate key do aeródromo (gerada por monotonically_increasing_id)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.sigla_icao_aeroporto IS 'Código OACI de 4 letras do aeródromo (chave natural)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.aerodromo IS 'Nome do aeródromo no formato NOME (CODIGO_OACI), ex: GUARULHOS (SBGR)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.nome IS 'Denominação oficial do aeródromo';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.municipio IS 'Município onde se localiza o aeródromo';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.uf IS 'Unidade da federação do município do aeródromo';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.latgeopoint IS 'Latitude do ponto de referência do aeródromo em formato decimal';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.longeopoint IS 'Longitude do ponto de referência do aeródromo em formato decimal';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.altitude IS 'Altitude do ponto mais elevado na área de pouso, em metros';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_aerodromos.situacao IS 'Situação cadastral do aeródromo: CADASTRADO, INTERDITADO';

COMMENT ON TABLE mvp_engenharia_dados.gold.dim_calendario IS 'Dimensão calendário derivada dinamicamente do range de datas (MIN/MAX) da silver_vra. Cobre de janeiro/2023 até o último dia do mês da data máxima. Contém atributos temporais para análise de sazonalidade e tendências.';

COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.sk_data IS 'Surrogate key da data (inteiro no formato yyyyMMdd, ex: 20230101)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.data IS 'Data (tipo date)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.ano IS 'Ano da data';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.mes IS 'Número do mês (1 a 12)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.nome_mes IS 'Nome do mês em português (Janeiro, Fevereiro, etc.)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.ano_mes IS 'Ano e mês no formato yyyy-MM (ex: 2023-01)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.dia IS 'Dia do mês (1 a 31)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.dia_semana IS 'Número do dia da semana (1=Domingo, 7=Sábado)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.nome_dia_semana IS 'Nome do dia da semana em português (Domingo, Segunda-feira, etc.)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.trimestre IS 'Trimestre (1 a 4)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.nome_trimestre IS 'Rótulo do trimestre no formato T# AAAA (ex: T1 2023)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.semana_ano IS 'Número da semana no ano (ISO)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.fim_de_semana IS 'Indica se a data é sábado ou domingo';
COMMENT ON COLUMN mvp_engenharia_dados.gold.dim_calendario.dia_util IS 'Indica se a data é dia útil (segunda a sexta)';

COMMENT ON TABLE mvp_engenharia_dados.gold.fato_voos IS 'Tabela fato de voos realizados domésticos de passageiros no Brasil. Originada da silver_vra (após filtro de voos de carga com numero_de_assentos = 0), com INNER JOIN para dim_empresas, dim_aerodromos (origem e destino) e dim_calendario. Cobertura de 100% dos registros da silver_vra.';

COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.sk_voo IS 'Surrogate key do voo (gerada por monotonically_increasing_id)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.sk_empresa IS 'FK para dim_empresas.sk_empresa';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.sk_data_partida IS 'FK para dim_calendario.sk_data (data da partida)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.sk_data_chegada IS 'FK para dim_calendario.sk_data (data da chegada)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.sk_aeroporto_origem IS 'FK para dim_aerodromos.sk_aerodromo (aeroporto de origem)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.sk_aeroporto_destino IS 'FK para dim_aerodromos.sk_aerodromo (aeroporto de destino)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.numero_voo IS 'Numeração do voo';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.modelo_equipamento IS 'Modelo do equipamento utilizado no voo no padrão ICAO';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.numero_de_assentos IS 'Quantidade de assentos da aeronave';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.partida_prevista IS 'Data e horário da partida prevista informada pela empresa aérea (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.partida_real IS 'Data e horário da partida realizada informada pela empresa aérea (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.chegada_prevista IS 'Data e horário da chegada prevista informada pela empresa aérea (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.chegada_real IS 'Data e horário da chegada realizada informada pela empresa aérea (horário de Brasília)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.situacao_partida IS 'Situação do voo na partida: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.situacao_chegada IS 'Situação do voo na chegada: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.pontual_partida IS 'TRUE quando situacao_partida é Antecipado ou Pontual (voo não atrasou na partida)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.pontual_chegada IS 'TRUE quando situacao_chegada é Antecipado ou Pontual (voo não atrasou na chegada)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.atraso_partida_min IS 'Atraso na partida em minutos (valor negativo indica adiantamento)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.atraso_chegada_min IS 'Atraso na chegada em minutos (valor negativo indica adiantamento)';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.descricao_etapa IS 'Descrição da etapa do voo: Etapa Regular, Etapa de Voo Charter, Etapa de Voo de Fretamento, etc.';
COMMENT ON COLUMN mvp_engenharia_dados.gold.fato_voos.referencia IS 'Mês de referência da publicação ANAC (período ao qual as decolagens previstas/realizadas pertencem)';

In [0]:
print("=" * 60)
print("VERIFICAÇÃO DA CAMADA GOLD")
print("=" * 60)

col_counts = {r.table_name: r.n_cols for r in spark.sql("""
    SELECT table_name, COUNT(*) AS n_cols
    FROM mvp_engenharia_dados.information_schema.columns
    WHERE table_schema = 'gold'
    GROUP BY table_name
""").collect()}

for tabela in ["dim_empresas", "dim_aerodromos", "dim_calendario", "fato_voos"]:
    n_rows = spark.table(f"mvp_engenharia_dados.gold.{tabela}").count()
    print(f"{tabela:35s} {n_rows:>12,d} registros  ({col_counts[tabela]} colunas)")

print()
print("Amostra fato_voos (primeiros voos):")
display(spark.table("mvp_engenharia_dados.gold.fato_voos").select("sk_voo", "sk_empresa", "sk_aeroporto_origem", "sk_aeroporto_destino", "situacao_partida", "situacao_chegada").limit(5))

print("\nConstraints em fato_voos:")
display(spark.sql("""
SELECT constraint_name, constraint_type
FROM mvp_engenharia_dados.information_schema.table_constraints
WHERE table_name = 'fato_voos'
  AND table_schema = 'gold'
ORDER BY constraint_type, constraint_name
"""))

## Resumo da Camada Gold

### Modelo Star Schema

| Tabela | Tipo | Registros | Descrição |
| --- | --- | --- | --- |
| `gold.dim_empresas` | Dimensão | 16 | Empresas aéreas com surrogate key (sk_empresa) |
| `gold.dim_aerodromos` | Dimensão | 504 | Aeródromos brasileiros com surrogate key (sk_aerodromo) |
| `gold.dim_calendario` | Dimensão | 1.339 | Calendário derivado do range de datas (partida e chegada) da silver_vra, com ano, mês, dia da semana, trimestre, semana e flags |
| `gold.fato_voos` | Fato | 2.732.331 | Tabela fato com FKs para dim_empresas, dim_aerodromos (origem/destino) e dim_calendario (partida/chegada) |

### Constraints (PK e FK)

| Tabela | Constraint | Tipo | Coluna(s) |
| --- | --- | --- | --- |
| `dim_empresas` | `pk_dim_empresas` | PRIMARY KEY | `sk_empresa` |
| `dim_aerodromos` | `pk_dim_aerodromos` | PRIMARY KEY | `sk_aerodromo` |
| `dim_calendario` | `pk_dim_calendario` | PRIMARY KEY | `sk_data` |
| `fato_voos` | `pk_fato_voos` | PRIMARY KEY | `sk_voo` |
| `fato_voos` | `fk_fato_empresa` | FOREIGN KEY | `sk_empresa` → `dim_empresas.sk_empresa` |
| `fato_voos` | `fk_fato_aeroporto_origem` | FOREIGN KEY | `sk_aeroporto_origem` → `dim_aerodromos.sk_aerodromo` |
| `fato_voos` | `fk_fato_aeroporto_destino` | FOREIGN KEY | `sk_aeroporto_destino` → `dim_aerodromos.sk_aerodromo` |
| `fato_voos` | `fk_fato_data_partida` | FOREIGN KEY | `sk_data_partida` → `dim_calendario.sk_data` |
| `fato_voos` | `fk_fato_data_chegada` | FOREIGN KEY | `sk_data_chegada` → `dim_calendario.sk_data` |

### Colunas de pontualidade normalizada

A fato_voos possui duas colunas booleanas que simplificam agregações de Índice de Pontualidade:
- `pontual_partida` (boolean): TRUE quando `situacao_partida` é Antecipado ou Pontual
- `pontual_chegada` (boolean): TRUE quando `situacao_chegada` é Antecipado ou Pontual

### Categorias de pontualidade

- **Pontual:** Antecipado + Pontual (voo adiantou ou cumpriu o horário previsto)
- **Atrasado:** Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240 (minutos)
- **Cobertura:** 100% dos registros da silver_vra com match em todas as dimensões (validado no notebook 04-camada-silver)

### Dicionário de Dados

#### `dim_empresas` — Dimensão de empresas aéreas (16 registros, 8 colunas)

| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `sk_empresa` | bigint | Surrogate key da empresa (PK) |
| `sigla_icao_empresa` | string | Designador ICAO de 3 letras da empresa aérea (chave natural) |
| `empresa` | string | Nome curto no formato PRIMEIRA_PALAVRA (ICAO), ex: GOL (GLO) |
| `razao` | string | Razão social da empresa aérea |
| `cidade` | string | Cidade onde a empresa é sediada |
| `uf` | string | Unidade da Federação da sede da empresa |
| `ativa` | string | Status no cadastro da ANAC (ATIVA/INATIVA) |
| `estrangeira` | string | Código designador internacional (ex: WD, 2S). "-" quando não possui |

#### `dim_aerodromos` — Dimensão de aeródromos (504 registros, 10 colunas)

| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `sk_aerodromo` | bigint | Surrogate key do aeródromo (PK) |
| `sigla_icao_aeroporto` | string | Código OACI de 4 letras (chave natural) |
| `aerodromo` | string | Nome no formato NOME (CODIGO_OACI), ex: GUARULHOS (SBGR) |
| `nome` | string | Denominação oficial do aeródromo |
| `municipio` | string | Município onde se localiza o aeródromo |
| `uf` | string | Unidade da Federação do município |
| `latgeopoint` | double | Latitude em graus decimais (WGS-84) |
| `longeopoint` | double | Longitude em graus decimais (WGS-84) |
| `altitude` | double | Altitude do ponto mais elevado, em metros |
| `situacao` | string | Situação cadastral: CADASTRADO, INTERDITADO |

#### `dim_calendario` — Dimensão de tempo (1.339 registros, 14 colunas)

| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `data` | date | Data (tipo date) |
| `sk_data` | int | Surrogate key no formato yyyyMMdd (PK) |
| `ano` | int | Ano da data |
| `mes` | int | Número do mês (1 a 12) |
| `nome_mes` | string | Nome do mês em português (Janeiro, Fevereiro, etc.) |
| `ano_mes` | string | Ano e mês no formato yyyy-MM (ex: 2023-01) |
| `dia` | int | Dia do mês (1 a 31) |
| `dia_semana` | int | Número do dia da semana (1=Domingo, 7=Sábado) |
| `nome_dia_semana` | string | Nome do dia da semana em português |
| `trimestre` | int | Trimestre (1 a 4) |
| `nome_trimestre` | string | Rótulo no formato T# AAAA (ex: T1 2023) |
| `semana_ano` | int | Número da semana no ano (ISO) |
| `fim_de_semana` | boolean | TRUE se sábado ou domingo |
| `dia_util` | boolean | TRUE se segunda a sexta |

#### `fato_voos` — Tabela fato de voos (2.732.331 registros, 21 colunas)

| Coluna | Tipo | Descrição |
| --- | --- | --- |
| `sk_voo` | bigint | Surrogate key do voo (PK) |
| `sk_empresa` | bigint | FK para dim_empresas.sk_empresa |
| `sk_data_partida` | int | FK para dim_calendario.sk_data (data da partida) |
| `sk_data_chegada` | int | FK para dim_calendario.sk_data (data da chegada) |
| `sk_aeroporto_origem` | bigint | FK para dim_aerodromos.sk_aerodromo (origem) |
| `sk_aeroporto_destino` | bigint | FK para dim_aerodromos.sk_aerodromo (destino) |
| `numero_voo` | string | Numeração do voo atribuída pela empresa |
| `modelo_equipamento` | string | Modelo da aeronave no padrão ICAO (ex: B738, A320) |
| `numero_de_assentos` | int | Quantidade de assentos da aeronave |
| `partida_prevista` | timestamp | Data e horário previstos para a decolagem (horário de Brasília) |
| `partida_real` | timestamp | Data e horário reais da decolagem (horário de Brasília) |
| `chegada_prevista` | timestamp | Data e horário previstos para o pouso (horário de Brasília) |
| `chegada_real` | timestamp | Data e horário reais do pouso (horário de Brasília) |
| `situacao_partida` | string | Situação na partida: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240 |
| `situacao_chegada` | string | Situação na chegada: Antecipado, Pontual, Atraso 30-60, Atraso 60-120, Atraso 120-240, Atraso > 240 |
| `pontual_partida` | boolean | TRUE quando situacao_partida é Antecipado ou Pontual |
| `pontual_chegada` | boolean | TRUE quando situacao_chegada é Antecipado ou Pontual |
| `atraso_partida_min` | double | Atraso na partida em minutos (negativo = adiantamento) |
| `atraso_chegada_min` | double | Atraso na chegada em minutos (negativo = adiantamento) |
| `descricao_etapa` | string | Tipo de etapa: Etapa Regular, Etapa de Voo Charter, etc. |
| `referencia` | date | Mês de referência da publicação ANAC |